# Customer support ticket router
Run the cells in order in Google Colab. This LangGraph example routes tickets using keyword rules, with no LLM or API key. The state contains the ticket, category, and reply.

Flow: start → classify → billing or technical → end. Billing keywords take priority; tickets without billing keywords go to technical support.

In [ ]:
%pip install -q langgraph


In [ ]:
import re
from typing import Literal, TypedDict

from IPython.display import Image, display
from langgraph.graph import END, START, StateGraph


class TicketState(TypedDict):
    ticket: str
    category: Literal['billing', 'technical'] | None
    reply: str


def classify(state: TicketState) -> dict:
    words = set(re.findall(r"[a-z]+", state['ticket'].lower()))
    billing_keywords = {
        'bill', 'billing', 'invoice', 'payment', 'charge',
        'charged', 'refund', 'subscription',
    }
    category = 'billing' if words & billing_keywords else 'technical'
    return {'category': category}


def route_ticket(state: TicketState) -> Literal['billing', 'technical']:
    if state['category'] == 'billing':
        return 'billing'
    return 'technical'


def billing(state: TicketState) -> dict:
    return {
        'reply': 'Our billing team will review your account and payment details.'
    }


def technical(state: TicketState) -> dict:
    return {
        'reply': 'Please restart the app and share any error message with technical support.'
    }


In [ ]:
builder = StateGraph(TicketState)
builder.add_node('classify', classify)
builder.add_node('billing', billing)
builder.add_node('technical', technical)

builder.add_edge(START, 'classify')
builder.add_conditional_edges(
    'classify',
    route_ticket,
    {'billing': 'billing', 'technical': 'technical'},
)
builder.add_edge('billing', END)
builder.add_edge('technical', END)

graph = builder.compile()


## Draw the graph
This cell uses the online Mermaid rendering service to display the graph. Only the graph structure is sent for rendering, not ticket state.

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))


## Try both routes
The invoice ticket follows the billing path. The app crash ticket follows the technical path. Each invocation starts with fresh state and prints the final category and reply.

In [ ]:
sample_tickets = [
    'I was charged twice on my latest invoice. Can I get a refund?',
    'The app crashes whenever I try to sign in.',
]

for ticket in sample_tickets:
    result = graph.invoke({
        'ticket': ticket,
        'category': None,
        'reply': '',
    })
    print(f"Ticket: {result['ticket']}")
    print(f"Category: {result['category']}")
    print(f"Reply: {result['reply']}")
    print()
